In [2]:
from google.colab import drive
drive.mount('/content/drive') 

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
os.chdir('/content/drive/MyDrive/DL_Series/llama2_code')
!cd /content/drive/MyDrive/DL_Series/llama2_code/llama2/llama && pip install -r requirements.txt

In [4]:
!cd /content/drive/MyDrive/DL_Series/llama2_code/llama2/llama && python model.py

Model initialized successfully. Running smoke test...
Transformer(
  (tok_embeddings): Embedding(1000, 128)
  (layers): ModuleList(
    (0-1): 2 x EncoderBlock(
      (attention): SelfAttention(
        (wq): Linear(in_features=128, out_features=128, bias=False)
        (wk): Linear(in_features=128, out_features=64, bias=False)
        (wv): Linear(in_features=128, out_features=64, bias=False)
        (wo): Linear(in_features=128, out_features=128, bias=False)
      )
      (ffn): FeedForward(
        (w1): Linear(in_features=128, out_features=512, bias=False)
        (w2): Linear(in_features=512, out_features=128, bias=False)
        (w3): Linear(in_features=128, out_features=512, bias=False)
      )
      (attention_norm): RMSNorm()
      (ffn_norm): RMSNorm()
    )
  )
  (norm): RMSNorm()
  (output): Linear(in_features=128, out_features=1000, bias=False)
)
Testing forward pass with dummy tokens...
All shape checks passed.


In [6]:
import os
size_gb = os.path.getsize("/content/drive/MyDrive/DL_Series/llama2_code/Llama2_7b_weights/consolidated.00.pth") / 1e9
print(f"{size_gb:.1f} GB")

13.5 GB


In [1]:
import shutil, os, time

src = "/content/drive/MyDrive/DL_Series/llama2_code/Llama2_7b_weights/consolidated.00.pth"
dst = "/content/consolidated.00.pth"

if not os.path.exists(dst):
    print("Copying to local SSD...")
    t0 = time.time()
    shutil.copy(src, dst)
    print(f"Done in {time.time()-t0:.1f}s")
else:
    print("Already cached locally, skipping copy.")

Already cached locally, skipping copy.


In [2]:
import torch, os

src = "/content/drive/MyDrive/DL_Series/llama2_code/Llama2_7b_weights/consolidated.00.pth"
dst = "/content/consolidated.00.pth"

# 1. Size matches
src_size = os.path.getsize(src)
dst_size = os.path.getsize(dst)
assert src_size == dst_size, f"Size mismatch: {src_size} vs {dst_size}"
print(f"Size OK: {dst_size/1e9:.1f} GB")

# 2. Loads without error and has expected keys
ckpt = torch.load(dst, map_location="cpu", mmap=True)
print(f"Keys: {len(ckpt)} tensors")
print("First few keys:", list(ckpt.keys())[:5])
del ckpt
print("Checkpoint OK")


Size OK: 13.5 GB
Keys: 292 tensors
First few keys: ['tok_embeddings.weight', 'norm.weight', 'output.weight', 'layers.0.attention.wq.weight', 'layers.0.attention.wk.weight']
Checkpoint OK


In [3]:
import shutil
# params.json is tiny — copy it alongside the cached .pth
shutil.copy(
    "/content/drive/MyDrive/DL_Series/llama2_code/Llama2_7b_weights/params.json",
    "/content/params.json"
)
print("params.json copied")

shutil.copy(
    "/content/drive/MyDrive/DL_Series/llama2_code/Llama2_7b_weights/tokenizer.model",
    "/content/tokenizer.model"
)
print("tokenizer.model copied")


params.json copied
tokenizer.model copied


In [4]:
import os, json, torch
os.chdir('/content/drive/MyDrive/DL_Series/llama2_code/llama2/llama')


In [5]:
dst = "/content/consolidated.00.pth"
ckpt = torch.load(dst, map_location="cpu", mmap=True)
print(f"Checkpoint loaded with {len(ckpt)} tensors")

Checkpoint loaded with 292 tensors


In [6]:
from typing import Optional
import torch
import time
from pathlib import Path
import glob as _glob
import json
from sentencepiece import SentencePieceProcessor
from tqdm import tqdm

from org_model import ModelArgs, Transformer


class LLaMA:

    def __init__(self, model: Transformer, tokenizer: SentencePieceProcessor, model_args: ModelArgs):
        self.model = model
        self.tokenizer = tokenizer
        self.args = model_args

    @staticmethod
    def build(checkpoints_dir: str, tokenizer_path: str, load_model: bool, max_seq_len: int, max_batch_size: int, device: str):
        prev_time = time.time()
        
        if load_model:
            checkpoints = sorted(Path(checkpoints_dir).glob("*.pth"))
            print("Checkpoint files found:", checkpoints)
            assert len(checkpoints) > 0, f"no checkpoint files found in {checkpoints_dir}"
            ckpt_path = checkpoints[0]
            print(f'Loading checkpoint "{ckpt_path}"')
            # checkpoint = torch.load(ckpt_path, map_location="cpu")
            checkpoint = torch.load(ckpt_path, map_location="cpu", mmap=True)
            print(f"Loaded checkpoint in {time.time() - prev_time:.2f}s")
            prev_time = time.time()
        with open(Path(checkpoints_dir) / "params.json", "r") as f:
            params = json.loads(f.read())

        model_args: ModelArgs = ModelArgs(
            max_seq_len=max_seq_len,
            max_batch_size=max_batch_size,
            device=device,
            **params
        )
        print(f"Model args: {model_args}")
        
        print(f"Loading tokenizer from {tokenizer_path}")
        tokenizer = SentencePieceProcessor()
        tokenizer.load(tokenizer_path)
        model_args.vocab_size = tokenizer.vocab_size()
        print("Tokenizer loaded with vocab size", model_args.vocab_size)
        
        print("Setting default tensor type and initializing model...")
        
        # if device == "cuda":
        #     torch.set_default_tensor_type(torch.cuda.HalfTensor)
        # else:
        #     torch.set_default_tensor_type(torch.BFloat16Tensor)
            
        torch.set_default_dtype(torch.float16)  # dtype only, no device
        torch.set_default_device("cpu")         # initialize on CPU
        
        print("Initializing model...")
        model = Transformer(model_args)
        
        print("Model initialized")
        
        print("Loading state dict into model...")
        if load_model:
            print(f"Checkpoint keys: {len(checkpoint)} tensors")
            print("checkpoint keys:", list(checkpoint.keys()))
            # The only unmatched key in the checkpoint is rope.freqs. Remove it
            del checkpoint['rope.freqs']
            model.load_state_dict(checkpoint, strict=True)
            print(f"Loaded state dict in {time.time() - prev_time:.2f}s")
        
        model = model.to(device)  # move AFTER weights are loaded (avoids double allocation)

        return LLaMA(model, tokenizer, model_args)

    def text_completion(self, prompts: list[str], temperature: float = 0.6, top_p: float = 0.9, max_gen_len: Optional[int] = None):
        if max_gen_len is None:
            max_gen_len = self.args.max_seq_len - 1
        # Convert each prompt into tokens
        prompt_tokens = [self.tokenizer.encode(prompt, out_type=int, add_bos=True, add_eos=False) for prompt in prompts]
        # Make sure the batch size is not too large
        batch_size = len(prompt_tokens)
        assert batch_size <= self.args.max_batch_size, f"batch size must be less than or equal to {self.args.max_batch_size}"
        max_prompt_len = max(len(prompt) for prompt in prompt_tokens)
        # Make sure the prompt length is not larger than the maximum sequence length
        assert max_prompt_len <= self.args.max_seq_len, f"prompt length must be less than or equal to {self.args.max_seq_len}"
        total_len = min(self.args.max_seq_len, max_gen_len + max_prompt_len)

        # Create the list that will contain the generated tokens, along with the initial prompt tokens
        pad_id = self.tokenizer.pad_id()
        tokens = torch.full((batch_size, total_len), pad_id, dtype=torch.long, device=device)
        for k, t in enumerate(prompt_tokens):
            # Populate the initial tokens with the prompt tokens
            tokens[k, : len(t)] = torch.tensor(t, dtype=torch.long, device=device)
        
        eos_reached = torch.tensor([False] * batch_size, device=device)
        prompt_tokens_mask = tokens != pad_id # True if the token is a prompt token, False otherwise
        cur_iterator = tqdm(range(1, total_len), desc="Generating tokens")
        for cur_pos in cur_iterator:
            with torch.no_grad():
                logits = self.model.forward(tokens[:, cur_pos-1:cur_pos], cur_pos)
            if temperature > 0:
                # The temperature is applied before the softmax
                probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
                next_token = self._sample_top_p(probs, top_p)
            else:
                # Greedily select the token with the max probability
                next_token = torch.argmax(logits[:, -1], dim=-1)

            next_token = next_token.reshape(-1)
            # Only replace token if it is a padding token
            next_token = torch.where(prompt_tokens_mask[:, cur_pos], tokens[:, cur_pos], next_token)
            tokens[:, cur_pos] = next_token
            # EOS is reached only if we found an EOS token for a padding position
            eos_reached |= (~prompt_tokens_mask[:, cur_pos]) & (next_token == self.tokenizer.eos_id)
            if all(eos_reached):
                break

        out_tokens = []
        out_text = []
        for prompt_index, current_prompt_tokens in enumerate(tokens.tolist()):
            # Cut to the EOS token, if present
            if self.tokenizer.eos_id in current_prompt_tokens:
                eos_idx = current_prompt_tokens.index(self.tokenizer.eos_id)
                current_prompt_tokens = current_prompt_tokens[:eos_idx]
            out_tokens.append(current_prompt_tokens)
            out_text.append(self.tokenizer.decode(current_prompt_tokens))
        return (out_tokens, out_text)
    
    def _sample_top_p(self, probs, p):
        # (B, vocab_size)
        probs_sort, probs_idx = torch.sort(probs, dim=-1, descending=True)
        # (B, vocab_size)
        probs_sum = torch.cumsum(probs_sort, dim=-1)
        # (B, vocab_size)
        # (Substracting "probs_sort" shifts the cumulative sum by 1 position to the right before masking)
        mask = probs_sum - probs_sort > p 
        # Zero out all the probabilities of tokens that are not selected by the Top P
        probs_sort[mask] = 0.0 
        # Redistribute the probabilities so that they sum up to 1.
        probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))
        # Sample a token (its index) from the top p distribution
        next_token = torch.multinomial(probs_sort, num_samples=1)
        # Get the token position in the vocabulary corresponding to the sampled index
        next_token = torch.gather(probs_idx, -1, next_token) 
        return next_token

In [7]:
if __name__ == '__main__':
    torch.manual_seed(0)

    allow_cuda = True
    device = 'cuda' if torch.cuda.is_available() and allow_cuda else 'cpu'

    prompts = [
        "Machine learning is ",
    ]
    
    checkpoints_dir = "/content/"
    
    # dst = "/content/consolidated.00.pth"
    # ckpt = torch.load(dst, map_location="cpu", mmap=True)
    # print(f"Checkpoint loaded with {len(ckpt)} tensors")
    
    model = LLaMA.build(
        checkpoints_dir=checkpoints_dir,
        tokenizer_path=f"{checkpoints_dir}/tokenizer.model",
        load_model=True,
        max_seq_len=1024,
        max_batch_size=len(prompts),
        device=device
    )
    print("All OK")
    
    # out_tokens, out_texts = (model.text_completion(prompts, max_gen_len=64))
    # assert len(out_texts) == len(prompts)
    # for i in range(len(out_texts)):
    #     print(f'{out_texts[i]}')
    #     print('-' * 50)

Checkpoint files found: [PosixPath('/content/consolidated.00.pth')]
Loading checkpoint "/content/consolidated.00.pth"
Loaded checkpoint in 0.04s
Model args: ModelArgs(dim=4096, n_layers=32, n_heads=32, n_kv_heads=None, vocab_size=-1, multiple_of=256, ffn_dim_multiplier=None, norm_eps=1e-05, max_batch_size=1, max_seq_len=1024, device='cpu')
Loading tokenizer from /content//tokenizer.model
Tokenizer loaded with vocab size 32000
Setting default tensor type and initializing model...
Initializing model...


: 

: 

: 